## 🥈 Silver Layer

In the Silver Layer, we will perform data cleaning and transformation to prepare the data for analysis. This includes handling missing values, converting data types, and creating new features as needed.

In [0]:
# Execute the setup script to prepare the environment and load necessary libraries
%%capture
%run ./00-setup

In [ ]:
# Read the bronze table
df_bronze = spark.read.table("sandbox_prd.bronze_layer.streaming_history_user_spotify")

In [0]:
# List of columns to drop from the bronze layer that are not needed for analysis
drop_columns = [
    "audiobook_chapter_title",
    "audiobook_chapter_uri",
    "audiobook_title",
    "audiobook_uri",
    "conn_country",
    "incognito_mode",
    "ip_addr",
    "dt_ingestion",
    "source_file"
]

# Remove unnecessary columns to create the silver layer dataframe
df_silver = df_bronze.drop(*drop_columns)

In [0]:
# Create a new column for the hour of the day in Brazil timezone
v_hora_brasil = hour(from_utc_timestamp(col("ts"), "America/Sao_Paulo"))


# select and tranform the columns to create the final silver layer dataframe
df_silver = df_silver.select(
    # Rename the columns to have a consistent naming convention
    col("episode_name").alias("nm_episode_name"),
    col("episode_show_name").alias("nm_episode_show_name"),
    col("master_metadata_album_album_name").alias("nm_album_name"),
    col("master_metadata_album_artist_name").alias("nm_artist_name"),
    col("master_metadata_track_name").alias("nm_track_name"),
    col("ms_played").alias("qt_played_ms"),
    col("offline").alias("fl_offline"),
    col("offline_timestamp").alias("ts_offline"),
    col("platform").alias("ds_platform"),
    col("reason_end").alias("ds_reason_end"),
    col("reason_start").alias("ds_reason_start"),
    col("shuffle").alias("fl_shuffle"),
    col("skipped").alias("fl_skipped"),
    col("ts").cast("Timestamp").alias("ts_streaming"),
    col("spotify_episode_uri").alias("ds_spotify_episode_uri"),
    col("spotify_track_uri").alias("ds_spotify_track_uri"),
    v_hora_brasil.alias("nr_hora_brasil"),

    # Create a column for year and month in the format "YYYY-MM"
    concat(
        year(col("ts")).cast("String"),
        lit("-"),
        month(col("ts")).cast("String")
    ).alias("dt_ano_mes"),

    # Create a column for duration in seconds
    (col("ms_played")/1000).cast("Int").alias("ts_duration_seconds"),

    # Create a column for duration in minutes
    (col("ms_played")/1000/60).cast("Int").alias("ts_duration_minutes"),

    # Create a column for the day of the week in Portuguese
    when(dayofweek(col("ts")) == 1, "Domingo")
        .when(dayofweek(col("ts")) == 2, "Segunda-feira")
        .when(dayofweek(col("ts")) == 3, "Terça-feira")
        .when(dayofweek(col("ts")) == 4, "Quarta-feira")
        .when(dayofweek(col("ts")) == 5, "Quinta-feira")
        .when(dayofweek(col("ts")) == 6, "Sexta-feira")
        .when(dayofweek(col("ts")) == 7, "Sábado")
        .alias("ds_day_of_week"),
    
    # Create a column for the period of the day in Portuguese
    when(v_hora_brasil < 6, "Madrugada")
        .when(v_hora_brasil < 12, "Manhã")
        .when(v_hora_brasil < 18, "Tarde")
        .otherwise("Noite").alias("ds_periodo_dia"),

    # Create a column for the order of the period of the day (1 for madrugada, 2 for manhã, 3 for tarde, 4 for noite)
    when(v_hora_brasil < 6, 1)
    .when(v_hora_brasil < 12, 2)
    .when(v_hora_brasil < 18, 3)
    .otherwise(4).alias("nr_ordem_periodo"),


    # Create a column for the type of start based on the reason_start column
    when(col("reason_start") == "trackdone", "Reprodução Automática")
    .when(col("reason_start") == "clickrow", "Seleção Manual")
    .when(col("reason_start") == "appload", "Retomada App")
    .when(col("reason_start") == "playbtn", "Botão Play")
    .when(col("reason_start").isin("fwdbtn", "backbtn"), "Navegação (Pular/Voltar)")
    .when(col("reason_start") == "remote", "Controle Externo")
    .otherwise("Outros").alias("ds_tipo_inicio"),

    # Create a column for the type of end based on the reason_end column
    when(lower(col("platform")).contains("android"), "Android")
    .when(lower(col("platform")).contains("ios"), "iOS")
    .when(lower(col("platform")).contains("web"), "Web")
    .when(lower(col("platform")).contains("windows"), "Windows")
    .when(lower(col("platform")).contains("mac"), "Mac")
    .when(lower(col("platform")).contains("linux"), "Linux")
    .when(lower(col("platform")).contains("tv"), "TV")
    .when(lower(col("platform")).contains("echo_show_5"), "Echo_Show")
    .when(lower(col("platform")).contains("other"), "Outros")
    .otherwise("Não identificado").alias("ds_device_type"),


    # Create a column for the Spotify link based on the available metadata (track, album, or episode)
   when(col("master_metadata_track_name").isNotNull(), 
         concat(lit("https://open.spotify.com/track/"), col("spotify_track_uri"))
    )
    .when(col("master_metadata_album_album_name").isNotNull(), 
          concat(lit("https://open.spotify.com/album/"), col("spotify_track_uri"))
    )
    .when(col("episode_show_name").isNotNull(), 
          concat(lit("https://open.spotify.com/episode/"), col("spotify_track_uri"))
    )
    .otherwise(lit("Não identificado")).alias("ds_link_musica"),
    to_date(col("ts")).alias("dt_referencia"),
)


In [0]:
# Save the dataframe in Silver Layer
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(NAME_TABLE_SILVER)

print(f"✅ Carga concluída com sucesso em: {NAME_TABLE_SILVER}")